# ERA5-Land (Google Earth Engine) → LSTM — **tout le Maroc, 1990 → 2025**

Ce notebook Colab transforme **tous les GeoTIFF** ERA5-Land stockés sur Google
Drive en un dataset tabulaire *long* prêt pour nos modèles LSTM.

**Idée clé :** un GeoTIFF est une *carte* de forme `(lat, lon, bandes)` où
`bandes = jours × variables`. Un LSTM attend `(batch_size, timesteps, features)`.
On déplie donc la carte : **chaque pixel de grille = une « station »** avec sa
série temporelle, puis on découpe en fenêtres glissantes.

```
carte .tif                per-pixel séries          fenêtres LSTM
(lat, lon, jour, var)  →  N pixels × T jours   →   (batch, 7 jours, F features)
```

Le notebook traite les fichiers **un par un** (robuste mémoire) et écrit **un
parquet par .tif** dans un dossier de sortie sur le Drive.

## 1. Installation des dépendances

In [ ]:
# imagecodecs = décodage LZW des GeoTIFF ; pyarrow = parquet
!pip -q install imagecodecs tifffile pyarrow

## 2. Montage de Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Configuration

Adapte `DRIVE_DIR` si besoin. `OUT_DIR` recevra un parquet par fichier .tif.

In [ ]:
from pathlib import Path

# Dossier contenant les GeoTIFF ERA5-Land (1990 -> 2025)
DRIVE_DIR = "/content/drive/MyDrive/ERA5_land_Morocco_GeoTIFF_All_new continet depuis 1990 vers 2025"

# Dossier de sortie (créé s'il n'existe pas)
OUT_DIR = "/content/drive/MyDrive/ERA5_land_LSTM_parquet"

# Fenêtres temporelles du LSTM
INPUT_WINDOW  = 7      # jours d'entrée
OUTPUT_WINDOW = 7      # jours à prédire
TARGET        = "heat_index"

Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
print("Entrée :", DRIVE_DIR)
print("Sortie :", OUT_DIR)

## 4. Fonctions — lecture, physique, dépliage, fenêtres

Robuste à des jeux de bandes différents : chaque feature dérivée n'est calculée
que si ses variables sources sont présentes dans le .tif.

In [ ]:
import re, xml.etree.ElementTree as ET
import numpy as np
import pandas as pd
import tifffile

# ---------------------------------------------------------------------------
# 4.1 Lecture GeoTIFF -> cube (H, W, jours, vars) + coords + dates + noms
# ---------------------------------------------------------------------------
def read_era5_tif(tif_path):
    with tifffile.TiffFile(tif_path) as tif:
        page = tif.pages[0]
        raw = page.asarray()                     # (H, W, n_bands)
        md  = page.tags["GDAL_METADATA"].value
        scale = page.tags["ModelPixelScaleTag"].value
        tie   = page.tags["ModelTiepointTag"].value
    H, W, n_bands = raw.shape

    root = ET.fromstring(md)
    descs = [it.text for it in root.findall(".//Item") if it.get("role") == "description"]
    dates_raw, vars_raw = [], []
    for d in descs:
        m = re.match(r"(\d{8})_(.+)", d)
        dates_raw.append(m.group(1)); vars_raw.append(m.group(2))

    var_names = list(dict.fromkeys(vars_raw))
    n_vars = len(var_names)
    assert n_bands % n_vars == 0, f"{n_bands} bandes non divisibles par {n_vars} variables"
    n_days = n_bands // n_vars
    dates = [pd.Timestamp(d) for d in dict.fromkeys(dates_raw)]

    cube = raw.reshape(H, W, n_days, n_vars).astype(np.float32)
    sx, sy = scale[0], scale[1]
    x0, y0 = tie[3], tie[4]
    lons = x0 + (np.arange(W) + 0.5) * sx
    lats = y0 - (np.arange(H) + 0.5) * sy
    return cube, lats, lons, dates, var_names

# ---------------------------------------------------------------------------
# 4.2 Formules physiques (identiques à clean_gsod_data.py)
# ---------------------------------------------------------------------------
def relative_humidity(temp_c, dewp_c):
    a, b = 17.625, 243.04
    es = 6.112 * np.exp((a * temp_c) / (b + temp_c))
    e  = 6.112 * np.exp((a * dewp_c) / (b + dewp_c))
    return np.clip((e / es) * 100.0, 0.0, 100.0)

def heat_index_vec(temp_c, rh):
    T = temp_c * 9.0 / 5.0 + 32.0
    c1,c2,c3 = -42.379, 2.04901523, 10.14333127
    c4,c5,c6 = -0.22475541, -0.00683783, -0.05481717
    c7,c8,c9 = 0.00122874, 0.00085282, -0.00000199
    hi = (c1 + c2*T + c3*rh + c4*T*rh + c5*T*T + c6*rh*rh
          + c7*T*T*rh + c8*T*rh*rh + c9*T*T*rh*rh)
    adj_dry = ((13-rh)/4.0) * np.sqrt(np.clip((17-np.abs(T-95))/17.0, 0, None))
    hi = np.where((rh < 13) & (T >= 80) & (T <= 112), hi - adj_dry, hi)
    adj_wet = ((rh-85)/10.0) * ((87-T)/5.0)
    hi = np.where((rh > 85) & (T >= 80) & (T <= 87), hi + adj_wet, hi)
    hi_cold = 0.5 * (T + 61.0 + ((T-68.0)*1.2) + (rh*0.094))
    HI = np.where(T < 80, hi_cold, hi)
    return (HI - 32.0) * 5.0 / 9.0

def wind_chill_vec(temp_c, wind_ms):
    kmh = wind_ms * 3.6
    wc = 13.12 + 0.6215*temp_c - 11.37*np.power(kmh,0.16) + 0.3965*temp_c*np.power(kmh,0.16)
    return np.where((temp_c <= 10.0) & (kmh > 4.8), wc, temp_c)

def elevation_from_pressure(press_hpa, temp_c):
    P0 = 1013.25
    return (np.power(P0/press_hpa, 1.0/5.257) - 1.0) * (temp_c + 273.15) / 0.0065

# ---------------------------------------------------------------------------
# 4.3 Cube -> DataFrame long (un pixel terre = une station). Adaptatif.
# ---------------------------------------------------------------------------
def cube_to_long(cube, lats, lons, dates, var_names):
    idx = {v: i for i, v in enumerate(var_names)}
    H, W, D, V = cube.shape
    def raw(name):
        return cube[rows, cols, :, idx[name]] if name in idx else None

    # masque terre : 1er variable de température dispo, non-NaN
    temp_key = next((k for k in ("temperature_2m_max","temperature_2m",
                                 "temperature_2m_min") if k in idx), var_names[0])
    valid = ~np.isnan(cube[:, :, 0, idx[temp_key]])
    rows, cols = np.where(valid)
    n_pix = rows.size
    print(f"   pixels terre = {n_pix}/{H*W} ({100*n_pix/(H*W):.1f}%), jours = {D}")
    if n_pix == 0:
        return pd.DataFrame()

    out = {}
    # températures K -> C
    for src, dst in [("temperature_2m_max","tmax_c"), ("temperature_2m_min","tmin_c"),
                     ("temperature_2m","tmean_c"), ("dewpoint_temperature_2m","dewp_c"),
                     ("soil_temperature_level_1","soil_temp_c")]:
        a = raw(src)
        if a is not None: out[dst] = a - 273.15
    if "tmean_c" not in out and "tmax_c" in out and "tmin_c" in out:
        out["tmean_c"] = (out["tmax_c"] + out["tmin_c"]) / 2.0

    # pression Pa -> hPa
    sp = raw("surface_pressure")
    if sp is not None: out["press_hpa"] = sp / 100.0
    # rayonnement J/m2 -> MJ/m2
    ssr = raw("surface_solar_radiation_downwards_sum")
    if ssr is not None: out["ssr_mj_m2"] = ssr / 1.0e6
    # sol
    sw = raw("volumetric_soil_water_layer_1")
    if sw is not None: out["soil_water"] = sw
    # précipitation m -> mm (ERA5-Land: total_precipitation_sum en mètres)
    tp = raw("total_precipitation_sum")
    if tp is not None: out["precip_mm"] = tp * 1000.0
    # vent
    u, v = raw("u_component_of_wind_10m"), raw("v_component_of_wind_10m")
    if u is not None and v is not None:
        out["wind_speed_ms"] = np.sqrt(u**2 + v**2)
        out["wind_dir_deg"]  = (np.degrees(np.arctan2(-u, -v))) % 360.0

    # humidité / heat index / wind chill (si sources dispo)
    t_for_hi = out.get("tmax_c", out.get("tmean_c"))
    if t_for_hi is not None and "dewp_c" in out:
        out["humidity_pct"] = relative_humidity(t_for_hi, out["dewp_c"])
        out["heat_index"]   = heat_index_vec(t_for_hi, out["humidity_pct"])
    t_for_wc = out.get("tmean_c", out.get("tmin_c"))
    if t_for_wc is not None and "wind_speed_ms" in out:
        out["wind_chill"] = wind_chill_vec(t_for_wc, out["wind_speed_ms"])

    # altitude statique estimée depuis la pression (médiane annuelle par pixel)
    if "press_hpa" in out and "tmean_c" in out:
        elev = elevation_from_pressure(np.median(out["press_hpa"], axis=1),
                                       np.median(out["tmean_c"], axis=1))
        elev = np.clip(elev, 0.0, None)
    else:
        elev = np.zeros(n_pix, dtype=np.float32)

    # assemblage long
    pixel_id = rows * W + cols
    base = {
        "pixel_id":  np.repeat(pixel_id, D),
        "latitude":  np.round(np.repeat(lats[rows], D), 4),
        "longitude": np.round(np.repeat(lons[cols], D), 4),
        "elevation": np.round(np.repeat(elev, D), 1),
        "date":      np.tile(np.array(dates, dtype="datetime64[ns]"), n_pix),
    }
    for k, arr in out.items():
        base[k] = arr.ravel()
    df = pd.DataFrame(base).sort_values(["pixel_id","date"]).reset_index(drop=True)
    return df

# ---------------------------------------------------------------------------
# 4.4 DataFrame long -> tenseurs LSTM (batch, timesteps, features)
# ---------------------------------------------------------------------------
def make_sequences(df, features, target="heat_index",
                   input_window=7, output_window=7, group_col="pixel_id"):
    feats = df[features].to_numpy(dtype=np.float32)
    tgt   = df[target].to_numpy(dtype=np.float32)
    codes = df[group_col].to_numpy()
    bounds = np.flatnonzero(np.diff(codes)) + 1
    starts = np.concatenate(([0], bounds)); ends = np.concatenate((bounds, [len(df)]))
    win = input_window + output_window
    Xs, ys = [], []
    for s, e in zip(starts, ends):
        if e - s < win: continue
        for t in range(s, e - win + 1):
            Xs.append(feats[t:t+input_window])
            ys.append(tgt[t+input_window:t+win])
    if not Xs:
        return (np.empty((0,input_window,len(features)),np.float32),
                np.empty((0,output_window),np.float32))
    return np.stack(Xs), np.stack(ys)

print("Fonctions chargées.")

## 5. Lister les GeoTIFF du Drive

In [ ]:
import glob, os
tif_files = sorted(glob.glob(os.path.join(DRIVE_DIR, "**", "*.tif"), recursive=True))
print(f"{len(tif_files)} fichiers .tif trouvés :")
for f in tif_files[:50]:
    print("  ", os.path.basename(f), f"({os.path.getsize(f)/1e6:.0f} Mo)")
if len(tif_files) > 50:
    print(f"  ... (+{len(tif_files)-50})")

## 6. Traitement — un parquet par fichier

Chaque .tif est lu, déplié, converti puis sauvegardé en parquet dans `OUT_DIR`.
Les fichiers déjà traités sont **sautés** (reprise possible après coupure Colab).
On libère la mémoire (`del cube`) entre chaque fichier.

In [ ]:
import gc, traceback

summary = []
for i, tif in enumerate(tif_files, 1):
    stem = Path(tif).stem
    out_pq = os.path.join(OUT_DIR, f"{stem}.parquet")
    if os.path.exists(out_pq):
        print(f"[{i}/{len(tif_files)}] {stem}: déjà fait, saut.")
        continue
    print(f"[{i}/{len(tif_files)}] {stem}: lecture...")
    try:
        cube, lats, lons, dates, var_names = read_era5_tif(tif)
        print(f"   cube {cube.shape}, variables={var_names}")
        df = cube_to_long(cube, lats, lons, dates, var_names)
        del cube; gc.collect()
        if df.empty:
            print("   aucun pixel terre, ignoré."); continue
        df.to_parquet(out_pq, index=False)
        print(f"   -> {out_pq}  ({df.shape[0]} lignes, {os.path.getsize(out_pq)/1e6:.1f} Mo)")
        summary.append((stem, df.shape[0], list(df.columns)))
        del df; gc.collect()
    except Exception as e:
        print(f"   ERREUR sur {stem}: {e}")
        traceback.print_exc()

print("\nTerminé. Fichiers produits :")
for s in summary: print("  ", s[0], "->", s[1], "lignes")

## 7. Vérification + démonstration du format LSTM

On recharge un parquet et on construit les fenêtres `(batch, timesteps, features)`.

In [ ]:
import glob
pq_files = sorted(glob.glob(os.path.join(OUT_DIR, "*.parquet")))
assert pq_files, "Aucun parquet produit — vérifie DRIVE_DIR."
df = pd.read_parquet(pq_files[0])
print("Fichier :", os.path.basename(pq_files[0]))
print("Shape   :", df.shape, "| pixels :", df.pixel_id.nunique(), "| jours :", df.date.nunique())
print("Colonnes:", list(df.columns))
display(df.head())

# features disponibles pour le LSTM (on garde celles présentes)
wanted = ["latitude","longitude","elevation","tmax_c","dewp_c","humidity_pct",
          "wind_speed_ms","press_hpa","ssr_mj_m2","soil_water","heat_index"]
FEATURES = [c for c in wanted if c in df.columns]
print("\nFEATURES:", FEATURES)

# démo sur un sous-ensemble de pixels
sub = df[df.pixel_id.isin(df.pixel_id.unique()[:100])]
X, y = make_sequences(sub, FEATURES, target=TARGET,
                      input_window=INPUT_WINDOW, output_window=OUTPUT_WINDOW)
print(f"\nX {X.shape} = (batch_size, timesteps, n_features)")
print(f"y {y.shape} = (batch_size, horizon)")

## 8. (Optionnel) Concaténer toutes les années en un seul dataset

⚠️ Selon la couverture (continent, 1990→2025) le fichier combiné peut être
**très volumineux**. À n'exécuter que si la RAM/disque le permettent — sinon
entraîner le LSTM en streaming fichier par fichier.

In [ ]:
# import pandas as pd, glob, os
# parts = [pd.read_parquet(f) for f in sorted(glob.glob(os.path.join(OUT_DIR, "*.parquet")))]
# full = pd.concat(parts, ignore_index=True)
# full = full.sort_values(["pixel_id","date"]).reset_index(drop=True)
# full.to_parquet(os.path.join(OUT_DIR, "_ALL_YEARS.parquet"), index=False)
# print(full.shape)